# OpenRAW — unattended InvISP training on Kaggle

This one notebook works in two modes, picked automatically from what's attached (**Add Input**):

1. **Prepare** (nothing attached; set *Accelerator: None* — no GPU quota used): downloads the FiveK cameras below,
   preprocesses them into OpenRAW's compact training format within the 20 GB Kaggle keeps, and saves it.
   Then on this notebook's **Output** tab choose **New Dataset**.
2. **Train** (that dataset attached; set *Accelerator: GPU T4 x2* (both GPUs are used)): trains until shortly before the 12 h session
   limit and stops cleanly. To continue next session, also attach **this notebook's own latest output** —
   it resumes exactly where it stopped. `STATUS.txt` in the output says *DONE* or *RUN AGAIN*.

**TPU (experimental)**: with Accelerator *TPU VM*, training runs on every TPU core (8 on Kaggle) through
PyTorch/XLA, after checking that the cores can talk to each other (else one core). Uses Kaggle's separate
TPU quota. New and only tested on an emulated single core so far: watch the first session's log.

**Comparing models**: every training session ends by scoring all models (upstream's, OpenRAW's shipped ones and
this run's `best.pth`/`latest.pth`) on the held-out test images → `compare/compare.md` in the output.
`COMPARE_ONLY = True` does just that, without training (CPU is fine).

**More data — RAISE** (optional): attach the CSV from the RAISE download page and set `RAISE_CSV = True`;
that run prepares RAISE (CPU) instead. Make its output a dataset and attach it to training next to FiveK.

Always start runs with **Save Version → Save & Run All (Commit)** so they run in the background.
Settings needed: **Internet on**. A Kaggle **Secret** named `GITHUB_TOKEN` is optional (the repo is public).
Full guide: `docs/kaggle.md` in the repo.

In [ ]:
import time
SESSION_START = time.time()   # for the 12 h session budget -- keep this the first line

# ---- configuration ---------------------------------------------------------
TASK = "openraw-fivek"        # checkpoint name; keep it the same across sessions
INIT_FROM = None              # e.g. "openraw-fivek": start a NEW task from that run's best.pth (attach its output)
CAMERAS = "all"               # "all" (largest first, within the budget) or e.g. ["Nikon D700", "Canon EOS 5D"]
DATA_BUDGET_GB = 18           # Kaggle keeps 20 GB of output; leave room for checkpoints
EPOCHS = None                 # None = auto: ~195k steps, upstream InvISP's training budget
WORKERS = 3                   # data-loading processes (Kaggle VMs have 4 CPUs)
ACCELERATOR = "auto"          # "auto" (GPU if the session has one, else TPU), "gpu", or "tpu" (experimental)
EXTRA_TRAIN_ARGS = []         # more train.py options, e.g. ["--eval_every", "3"]
COMPARE_ONLY = False          # True: don't train, just score every model on the test images (CPU is fine)
AMP_CHECK_ONLY = False        # True: don't train, measure whether mixed precision (--amp) pays off on this GPU
# RAISE (non-commercial research use): set to True with the CSV from its download page attached,
# on a CPU session, to prepare RAISE instead of training. Its output becomes a second dataset
# to attach next to the FiveK one. RAISE_START continues where a previous part stopped.
RAISE_CSV = None
RAISE_CAMERAS = []            # e.g. ["D90"]; [] = all three
RAISE_START = 0
REPO_URL = "https://github.com/Swarce/OpenRAW.git"
BRANCH = "main"

In [ ]:
import os, subprocess, sys

# GitHub token from Kaggle Secrets (Add-ons -> Secrets -> GITHUB_TOKEN); never printed or saved
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:
    print("no GITHUB_TOKEN secret -- cloning without one (works only for a public repo):", type(e).__name__)

SRC = "/tmp/OpenRAW"
if not os.path.isdir(SRC):
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, SRC], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError("git clone failed: " + r.stderr.replace(token or "\0", "***"))
    subprocess.run(["git", "-C", SRC, "remote", "set-url", "origin", REPO_URL])  # drop the token from .git/config
print(subprocess.run(["git", "-C", SRC, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Kaggle already has torch, numpy and pillow; add what OpenRAW's training needs (not OpenCV: it would clash)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rawpy", "colour-demosaicing", "imagecodecs", "tifffile"], check=True)

# TPU sessions (experimental): training runs on PyTorch/XLA. Use the copy in Kaggle's TPU image if
# there is one, else install the release matching this torch.
import glob, importlib.util
_tpu = ACCELERATOR == "tpu" or (ACCELERATOR == "auto" and bool(
    os.environ.get("TPU_NAME") or os.environ.get("TPU_ACCELERATOR_TYPE") or glob.glob("/dev/accel*")))
if _tpu and importlib.util.find_spec("torch_xla") is None:
    import torch
    _v = torch.__version__.split("+")[0]
    print(f"TPU session: installing PyTorch/XLA {_v} ...")
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={_v}",
                         "-f", "https://storage.googleapis.com/libtpu-releases/index.html"])
    if _r.returncode:
        print(f"Could not install PyTorch/XLA for torch {_v}: no matching release. See docs/kaggle.md (TPU).")


In [ ]:
sys.path.insert(0, SRC + "/kaggle")
import kaggle_runner as kr

result = kr.run(task=TASK, cameras=CAMERAS, budget_gb=DATA_BUDGET_GB, epochs=EPOCHS,
                workers=WORKERS, session_start=SESSION_START,
                raise_csv=RAISE_CSV, raise_cameras=RAISE_CAMERAS, raise_start=RAISE_START,
                compare_only=COMPARE_ONLY, init_from=INIT_FROM, amp_check_only=AMP_CHECK_ONLY,
                extra_train_args=EXTRA_TRAIN_ARGS, accelerator=ACCELERATOR)
print("result:", result)